# 05 — Classificazione per similarità

## Obiettivo

Utilizzare gli embedding appresi dalla Siamese Network per associare
un nuovo sample alla classe tumorale più simile.

La classificazione non viene effettuata tramite un classificatore esterno,
ma direttamente nello spazio metrico appreso dall'encoder Siamese.

Il notebook è organizzato in due fasi principali:

1. confronto equo tra i migliori encoder Euclidean e Cosine utilizzando
   le stesse identiche coppie di validation;
2. classificazione multiclass tramite similarità tra l'embedding di un
   sample e le rappresentazioni di riferimento delle diverse classi.

## Modelli selezionati

### Euclidean V3
- Contrastive Loss
- margin = 1.25
- CNN V3 bias-free
- embedding = 128
- best validation ROC-AUC = 0.677272

### Cosine V3
- CosineEmbeddingLoss
- margin = 0.80
- CNN V3 bias-free
- embedding = 128
- best validation ROC-AUC = 0.659911

Entrambi gli encoder utilizzano embedding L2-normalizzati.

In [1]:
# ============================================================
# CELL 2 — IMPORT
# ============================================================

from pathlib import Path

import json
import random
import gc

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader,
)

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
)


print(
    "PyTorch:",
    torch.__version__
)

print(
    "CUDA disponibile:",
    torch.cuda.is_available()
)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

PyTorch: 2.12.0+cu126
CUDA disponibile: True
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


In [2]:
# ============================================================
# CELL 3 — PATH E CONFIGURAZIONE
# ============================================================

CURRENT_DIR = (
    Path.cwd()
    .resolve()
)


if CURRENT_DIR.name == "notebooks":

    PROJECT_ROOT = (
        CURRENT_DIR.parent
    )

else:

    PROJECT_ROOT = (
        CURRENT_DIR
    )


PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)


ARTIFACTS_DIR = (
    PROJECT_ROOT
    / "artifacts"
)


OUTPUT_DIR = (
    ARTIFACTS_DIR
    / "similarity_classification"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# DATA
# ============================================================

PAIR_CONFIG_PATH = (
    PROCESSED_DIR
    / "siamese_pair_config.json"
)


MANIFEST_PATH = (
    PROCESSED_DIR
    / "siamese_primary_manifest.tsv"
)


VAL_POOL_PATH = (
    PROCESSED_DIR
    / "siamese_val_pair_pool.tsv"
)


TEST_POOL_PATH = (
    PROCESSED_DIR
    / "siamese_test_pair_pool.tsv"
)


with open(
    PAIR_CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:

    PAIR_CONFIG = json.load(f)


K = int(
    PAIR_CONFIG["k"]
)


RANDOM_STATE = int(
    PAIR_CONFIG["random_state"]
)


VAL_PAIRS = int(
    PAIR_CONFIG["val_pairs"]
)


POSITIVE_PAIR_PROBABILITY = float(
    PAIR_CONFIG["positive_probability"]
)


BATCH_SIZE = 128

EMBEDDING_DIM = 128


FCGR_MEMMAP_PATH = (
    PROCESSED_DIR
    / "fcgr_cache"
    / f"fcgr_k{K}.npy"
)


FCGR_INDEX_PATH = (
    PROCESSED_DIR
    / "fcgr_cache"
    / f"fcgr_k{K}_index.tsv"
)


# ============================================================
# CHECKPOINT
# ============================================================

EUCLIDEAN_CHECKPOINT_PATH = (
    ARTIFACTS_DIR
    / "siamese_euclidean_v3"
    / "euclidean_v3_margin_1p25_best.pt"
)


COSINE_CHECKPOINT_PATH = (
    ARTIFACTS_DIR
    / "cosine_v3"
    / "cosine_v3_margin_0p80_best.pt"
)


print(
    "Euclidean checkpoint:",
    EUCLIDEAN_CHECKPOINT_PATH
)

print(
    "Esiste:",
    EUCLIDEAN_CHECKPOINT_PATH.exists()
)

print()

print(
    "Cosine checkpoint:",
    COSINE_CHECKPOINT_PATH
)

print(
    "Esiste:",
    COSINE_CHECKPOINT_PATH.exists()
)

Euclidean checkpoint: D:\Daria\Desktop\eccdna_fcgr_siamese\artifacts\siamese_euclidean_v3\euclidean_v3_margin_1p25_best.pt
Esiste: True

Cosine checkpoint: D:\Daria\Desktop\eccdna_fcgr_siamese\artifacts\cosine_v3\cosine_v3_margin_0p80_best.pt
Esiste: True


In [3]:
# ============================================================
# CELL 4 — DEVICE, METADATA E FCGR
# ============================================================

def set_seed(
    seed
):

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


set_seed(
    RANDOM_STATE
)


DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


if DEVICE.type == "cuda":

    torch.backends.cudnn.benchmark = True

    torch.set_float32_matmul_precision(
        "high"
    )


# ============================================================
# METADATA
# ============================================================

metadata = pd.read_csv(
    MANIFEST_PATH,
    sep="\t",
    dtype={
        "id": str
    }
)


val_pool = pd.read_csv(
    VAL_POOL_PATH,
    sep="\t",
    dtype={
        "id": str
    }
)


test_pool = pd.read_csv(
    TEST_POOL_PATH,
    sep="\t",
    dtype={
        "id": str
    }
)


train_metadata = (
    metadata[
        metadata[
            "split_cluster"
        ] == "train"
    ]
    .copy()
)


for dataframe in [
    metadata,
    train_metadata,
    val_pool,
    test_pool
]:

    dataframe[
        "class_id"
    ] = (
        dataframe[
            "class_id"
        ]
        .astype(int)
    )


# ============================================================
# FCGR
# ============================================================

fcgr_memmap = np.load(
    FCGR_MEMMAP_PATH,
    mmap_mode="r"
)


fcgr_index = pd.read_csv(
    FCGR_INDEX_PATH,
    sep="\t",
    dtype={
        "id": str
    }
)


id_to_fcgr_row = dict(
    zip(
        fcgr_index["id"],
        fcgr_index["fcgr_row"]
    )
)


print(
    "Device:",
    DEVICE
)

print(
    "Train:",
    len(train_metadata)
)

print(
    "Validation:",
    len(val_pool)
)

print(
    "Test:",
    len(test_pool)
)

print(
    "Numero classi:",
    metadata[
        "class_id"
    ].nunique()
)

print(
    "FCGR:",
    fcgr_memmap.shape
)

Device: cuda
Train: 126265
Validation: 12937
Test: 11070
Numero classi: 18
FCGR: (150272, 64, 64)


In [4]:
# ============================================================
# CELL 5 — ARCHITETTURA V3 COMUNE
# ============================================================

class FCGRCNNEncoderV3(
    nn.Module
):

    def __init__(
        self,
        embedding_dim=128
    ):

        super().__init__()


        self.features = nn.Sequential(

            nn.Conv2d(
                1,
                32,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.GroupNorm(
                8,
                32
            ),

            nn.ReLU(
                inplace=True
            ),


            nn.Conv2d(
                32,
                32,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.MaxPool2d(
                2
            ),


            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.GroupNorm(
                8,
                64
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.MaxPool2d(
                2
            ),


            nn.Conv2d(
                64,
                128,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.GroupNorm(
                8,
                128
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.MaxPool2d(
                2
            ),


            nn.Conv2d(
                128,
                128,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.GroupNorm(
                8,
                128
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.AdaptiveAvgPool2d(
                (
                    4,
                    4
                )
            )
        )


        self.embedding_head = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                128 * 4 * 4,
                256
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Linear(
                256,
                embedding_dim
            )
        )


    def forward(
        self,
        x
    ):

        x = self.features(
            x
        )

        z = self.embedding_head(
            x
        )

        z = F.normalize(
            z,
            p=2,
            dim=1,
            eps=1e-8
        )

        return z


class SiameseNetworkV3(
    nn.Module
):

    def __init__(
        self,
        embedding_dim=128
    ):

        super().__init__()

        self.encoder = (
            FCGRCNNEncoderV3(
                embedding_dim
            )
        )


    def forward(
        self,
        x1,
        x2
    ):

        batch_size = (
            x1.shape[0]
        )

        x = torch.cat(
            [
                x1,
                x2
            ],
            dim=0
        )

        z = self.encoder(
            x
        )

        return (
            z[:batch_size],
            z[batch_size:]
        )

In [6]:
# ============================================================
# CELL 6 — CARICAMENTO MODELLI
#          + REMAPPING CHECKPOINT COSINE
# ============================================================

assert (
    EUCLIDEAN_CHECKPOINT_PATH.exists()
), (
    "Checkpoint Euclidean non trovato."
)

assert (
    COSINE_CHECKPOINT_PATH.exists()
), (
    "Checkpoint Cosine non trovato."
)


# ============================================================
# LOAD CHECKPOINTS
# ============================================================

euclidean_checkpoint = torch.load(
    EUCLIDEAN_CHECKPOINT_PATH,
    map_location=DEVICE
)


cosine_checkpoint = torch.load(
    COSINE_CHECKPOINT_PATH,
    map_location=DEVICE
)


# ============================================================
# CREATE MODELS
# ============================================================

euclidean_model = (
    SiameseNetworkV3(
        embedding_dim=EMBEDDING_DIM
    )
    .to(DEVICE)
)


cosine_model = (
    SiameseNetworkV3(
        embedding_dim=EMBEDDING_DIM
    )
    .to(DEVICE)
)


# ============================================================
# EUCLIDEAN
#
# Il checkpoint Euclidean usa già "embedding_head".
# ============================================================

euclidean_model.load_state_dict(
    euclidean_checkpoint[
        "model_state_dict"
    ]
)


# ============================================================
# COSINE
#
# Nel notebook Cosine il blocco finale si chiamava
# "projection", mentre nell'encoder comune del notebook 05
# si chiama "embedding_head".
#
# L'architettura è la stessa:
# cambiamo solamente i nomi delle chiavi.
# ============================================================

cosine_state_dict_original = (
    cosine_checkpoint[
        "model_state_dict"
    ]
)


cosine_state_dict_remapped = {}


for key, value in (
    cosine_state_dict_original.items()
):

    new_key = (
        key.replace(
            "encoder.projection.",
            "encoder.embedding_head."
        )
    )

    cosine_state_dict_remapped[
        new_key
    ] = value


# Caricamento STRICT:
# se esiste un'altra incompatibilità vogliamo saperlo.
cosine_model.load_state_dict(
    cosine_state_dict_remapped,
    strict=True
)


# ============================================================
# EVALUATION MODE
# ============================================================

euclidean_model.eval()

cosine_model.eval()


# ============================================================
# INFO
# ============================================================

print("=" * 70)
print("MODELLI CARICATI")
print("=" * 70)


print()
print("EUCLIDEAN")

print(
    "Best epoch:",
    euclidean_checkpoint.get(
        "best_epoch"
    )
)

print(
    "Val AUC originale:",
    euclidean_checkpoint.get(
        "best_val_auc"
    )
)

print(
    "Margin:",
    euclidean_checkpoint.get(
        "margin"
    )
)


print()
print("COSINE")

print(
    "Best epoch:",
    cosine_checkpoint.get(
        "best_epoch"
    )
)

print(
    "Val AUC originale:",
    cosine_checkpoint.get(
        "best_val_auc"
    )
)

print(
    "Margin:",
    cosine_checkpoint.get(
        "margin"
    )
)


# ============================================================
# SANITY CHECK
# ============================================================

print()
print("=" * 70)
print("SANITY CHECK")
print("=" * 70)

print(
    "Euclidean parameters:",
    sum(
        p.numel()
        for p in euclidean_model.parameters()
    )
)

print(
    "Cosine parameters:",
    sum(
        p.numel()
        for p in cosine_model.parameters()
    )
)


assert (
    sum(
        p.numel()
        for p in euclidean_model.parameters()
    )
    ==
    sum(
        p.numel()
        for p in cosine_model.parameters()
    )
)


print()
print(
    "Architetture compatibili: OK"
)

MODELLI CARICATI

EUCLIDEAN
Best epoch: 38
Val AUC originale: 0.67727181092637
Margin: 1.25

COSINE
Best epoch: 46
Val AUC originale: 0.6599109638794127
Margin: 0.8

SANITY CHECK
Euclidean parameters: 807264
Cosine parameters: 807264

Architetture compatibili: OK


In [7]:
# ============================================================
# CELL 7 — ISPEZIONE LABEL
# ============================================================

print(
    "Colonne del manifest:"
)

print(
    metadata.columns.tolist()
)


print()
print(
    "Prime righe:"
)

display(
    metadata.head()
)


print()
print(
    "Distribuzione class_id:"
)

display(

    metadata[
        "class_id"
    ]
    .value_counts()
    .sort_index()
    .rename(
        "n_samples"
    )
    .to_frame()
)

Colonne del manifest:
['id', 'disease', 'disease_clean', 'disease_group', 'class_id', 'split_cluster', 'cluster_id', 'length', 'gc', 'n_fraction', 'source_db', 'tissue']

Prime righe:


,id,disease,disease_clean,disease_group,class_id,split_cluster,cluster_id,length,gc,n_fraction,source_db,tissue
0,CircleBaseV2_000726522,Gastric cancer,gastric cancer,cancer,0,train,c000360518,1726,0.335458,0.0,CircleBaseV2,Stomach
1,CircleBaseV2_000119689,Gastric cancer,gastric cancer,cancer,0,train,c000007603,1502,0.448069,0.0,CircleBaseV2,Stomach
2,CircleBaseV2_002112184,Gastric cancer,gastric cancer,cancer,0,train,c000185012,6595,0.370129,0.0,CircleBaseV2,Stomach
3,CircleBaseV2_001452203,Gastric cancer,gastric cancer,cancer,0,train,c000077332,1431,0.343117,0.0,CircleBaseV2,Stomach
4,CircleBaseV2_000827693,Gastric cancer,gastric cancer,cancer,0,train,c000382479,2421,0.397356,0.0,CircleBaseV2,Stomach



Distribuzione class_id:


,n_samples
class_id,
0,391239
1,73995
2,35122
3,34487
4,22933
5,20185
6,16315
7,15677
8,13456


In [8]:
# ============================================================
# CELL 8 — MAPPING CLASS_ID -> DISEASE
# ============================================================

class_mapping = (
    metadata[
        [
            "class_id",
            "disease_clean"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "class_id"
    )
    .reset_index(
        drop=True
    )
)


# Ogni class_id deve corrispondere
# a una sola disease_clean.
class_name_counts = (
    class_mapping
    .groupby(
        "class_id"
    )[
        "disease_clean"
    ]
    .nunique()
)


assert (
    class_name_counts.max()
    ==
    1
), (
    "Un class_id è associato "
    "a più disease_clean."
)


class_id_to_name = dict(
    zip(
        class_mapping[
            "class_id"
        ],

        class_mapping[
            "disease_clean"
        ]
    )
)


display(
    class_mapping
)


print()
print(
    "Numero classi:",
    len(class_id_to_name)
)

,class_id,disease_clean
0,0,gastric cancer
1,1,healthy
2,2,ovarian cancer
3,3,prostate cancer
4,4,colorectal cancer
5,5,lymphoma
6,6,hiv infectious disease
7,7,cervical adenocarcinoma
8,8,leukemia
9,9,primary pulmonary hypertension



Numero classi: 18


In [9]:
# ============================================================
# CELL 9 — COMMON VALIDATION PAIRS
#
# Queste coppie saranno identiche per:
# - Euclidean
# - Cosine
# ============================================================

COMMON_VAL_SEED = (
    RANDOM_STATE
    +
    50_000
)


rng = np.random.default_rng(
    COMMON_VAL_SEED
)


# ------------------------------------------------------------
# Preparazione class -> sample
# ------------------------------------------------------------

val_by_class = {

    int(class_id):
        group[
            [
                "id",
                "class_id"
            ]
        ]
        .reset_index(
            drop=True
        )

    for class_id, group
    in val_pool.groupby(
        "class_id"
    )
}


val_classes = np.array(
    sorted(
        val_by_class.keys()
    ),
    dtype=np.int64
)


common_pairs = []


for pair_index in range(
    VAL_PAIRS
):

    # ========================================================
    # FIRST SAMPLE
    # ========================================================

    class1_idx = int(
        rng.integers(
            0,
            len(val_classes)
        )
    )

    class1 = int(
        val_classes[
            class1_idx
        ]
    )


    group1 = (
        val_by_class[
            class1
        ]
    )


    i1 = int(
        rng.integers(
            0,
            len(group1)
        )
    )


    id1 = str(
        group1.iloc[
            i1
        ][
            "id"
        ]
    )


    # ========================================================
    # POSITIVE
    # ========================================================

    if (
        rng.random()
        <
        POSITIVE_PAIR_PROBABILITY
    ):

        if len(group1) < 2:

            raise ValueError(
                f"Classe {class1} "
                "con meno di due sample validation."
            )


        i2 = int(
            rng.integers(
                0,
                len(group1) - 1
            )
        )


        if i2 >= i1:
            i2 += 1


        class2 = class1

        id2 = str(
            group1.iloc[
                i2
            ][
                "id"
            ]
        )


        target = 1


    # ========================================================
    # NEGATIVE
    # ========================================================

    else:

        class2_idx = int(
            rng.integers(
                0,
                len(val_classes) - 1
            )
        )


        if (
            class2_idx
            >=
            class1_idx
        ):

            class2_idx += 1


        class2 = int(
            val_classes[
                class2_idx
            ]
        )


        group2 = (
            val_by_class[
                class2
            ]
        )


        i2 = int(
            rng.integers(
                0,
                len(group2)
            )
        )


        id2 = str(
            group2.iloc[
                i2
            ][
                "id"
            ]
        )


        target = 0


    common_pairs.append(
        {
            "pair_id":
                pair_index,

            "id1":
                id1,

            "id2":
                id2,

            "class1":
                class1,

            "class2":
                class2,

            "target":
                target
        }
    )


common_val_pairs_df = pd.DataFrame(
    common_pairs
)


print(
    "Common validation pairs:",
    len(common_val_pairs_df)
)

print(
    "Positive:",
    int(
        common_val_pairs_df[
            "target"
        ].sum()
    )
)

print(
    "Negative:",
    int(
        (
            common_val_pairs_df[
                "target"
            ] == 0
        ).sum()
    )
)


display(
    common_val_pairs_df.head()
)

Common validation pairs: 10000
Positive: 5000
Negative: 5000


,pair_id,id1,id2,class1,class2,target
0,0,eccDNABase_000887100,eccDNABase_000126200,14,3,0
1,1,CircleBaseV2_002938226,CircleBaseV2_002936220,4,4,1
2,2,eccDNABase_000006025,eccDNABase_000441400,9,8,0
3,3,eccDNABase_000572517,CircleBaseV2_000028430,12,10,0
4,4,eccDNABase_000441228,eccDNABase_000509043,8,8,1


In [10]:
# ============================================================
# CELL 10 — SAVE COMMON VALIDATION PAIRS
# ============================================================

COMMON_VAL_PAIRS_PATH = (
    OUTPUT_DIR
    /
    "common_validation_pairs.tsv"
)


common_val_pairs_df.to_csv(
    COMMON_VAL_PAIRS_PATH,
    sep="\t",
    index=False
)


print(
    "Salvate in:",
    COMMON_VAL_PAIRS_PATH
)

Salvate in: D:\Daria\Desktop\eccdna_fcgr_siamese\artifacts\similarity_classification\common_validation_pairs.tsv


In [11]:
# ============================================================
# CELL 11 — FIXED PAIR DATASET
# ============================================================

class FixedPairDataset(
    Dataset
):

    def __init__(
        self,
        pairs_df,
        fcgr_memmap,
        id_to_row
    ):

        self.pairs = (
            pairs_df
            .reset_index(
                drop=True
            )
        )

        self.fcgr_memmap = (
            fcgr_memmap
        )

        self.id_to_row = (
            id_to_row
        )


        # Conversione ID -> row
        # fatta una sola volta.
        self.row1 = (
            self.pairs[
                "id1"
            ]
            .map(
                self.id_to_row
            )
            .to_numpy(
                dtype=np.int64
            )
        )


        self.row2 = (
            self.pairs[
                "id2"
            ]
            .map(
                self.id_to_row
            )
            .to_numpy(
                dtype=np.int64
            )
        )


        self.targets = (
            self.pairs[
                "target"
            ]
            .to_numpy(
                dtype=np.float32
            )
        )


    def __len__(
        self
    ):

        return len(
            self.pairs
        )


    def _load(
        self,
        row
    ):

        x = np.array(
            self.fcgr_memmap[
                row
            ],
            dtype=np.float32,
            copy=True
        )

        return (
            torch.from_numpy(
                x
            )
            .unsqueeze(0)
        )


    def __getitem__(
        self,
        index
    ):

        return {

            "x1":
                self._load(
                    self.row1[
                        index
                    ]
                ),

            "x2":
                self._load(
                    self.row2[
                        index
                    ]
                ),

            "target":
                torch.tensor(
                    self.targets[
                        index
                    ],
                    dtype=torch.float32
                )
        }


common_val_dataset = FixedPairDataset(

    pairs_df=
        common_val_pairs_df,

    fcgr_memmap=
        fcgr_memmap,

    id_to_row=
        id_to_fcgr_row
)


common_val_loader = DataLoader(

    common_val_dataset,

    batch_size=
        BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=
        torch.cuda.is_available()
)


print(
    "Common val batches:",
    len(common_val_loader)
)

Common val batches: 79


In [12]:
# ============================================================
# CELL 12 — FAIR PAIRWISE COMPARISON
# ============================================================

def evaluate_common_pairs(
    model,
    loader
):

    model.eval()


    all_targets = []

    all_euclidean = []

    all_cosine = []


    with torch.no_grad():

        for batch in loader:

            x1 = (
                batch[
                    "x1"
                ]
                .to(
                    DEVICE,
                    non_blocking=True
                )
            )


            x2 = (
                batch[
                    "x2"
                ]
                .to(
                    DEVICE,
                    non_blocking=True
                )
            )


            targets = (
                batch[
                    "target"
                ]
                .cpu()
                .numpy()
            )


            with torch.autocast(

                device_type=
                    DEVICE.type,

                dtype=
                    torch.float16
                    if DEVICE.type == "cuda"
                    else torch.bfloat16,

                enabled=
                    DEVICE.type == "cuda"

            ):

                z1, z2 = model(
                    x1,
                    x2
                )


            euclidean_distance = (
                F.pairwise_distance(
                    z1,
                    z2,
                    p=2,
                    eps=1e-8
                )
                .float()
                .cpu()
                .numpy()
            )


            cosine_similarity = (
                F.cosine_similarity(
                    z1,
                    z2,
                    dim=1,
                    eps=1e-8
                )
                .float()
                .cpu()
                .numpy()
            )


            all_targets.append(
                targets
            )

            all_euclidean.append(
                euclidean_distance
            )

            all_cosine.append(
                cosine_similarity
            )


    targets = np.concatenate(
        all_targets
    ).astype(
        np.int64
    )


    distances = np.concatenate(
        all_euclidean
    )


    similarities = np.concatenate(
        all_cosine
    )


    # ========================================================
    # EUCLIDEAN METRICS
    # ========================================================

    d_pos = float(
        distances[
            targets == 1
        ].mean()
    )


    d_neg = float(
        distances[
            targets == 0
        ].mean()
    )


    d_gap = float(
        d_neg
        -
        d_pos
    )


    d_pooled_std = np.sqrt(

        0.5
        *
        (
            distances[
                targets == 1
            ].var()
            +
            distances[
                targets == 0
            ].var()
        )

        +
        1e-12
    )


    euclidean_d_prime = float(
        d_gap
        /
        d_pooled_std
    )


    euclidean_auc = float(
        roc_auc_score(
            targets,
            -distances
        )
    )


    # ========================================================
    # COSINE METRICS
    # ========================================================

    cos_pos = float(
        similarities[
            targets == 1
        ].mean()
    )


    cos_neg = float(
        similarities[
            targets == 0
        ].mean()
    )


    cos_gap = float(
        cos_pos
        -
        cos_neg
    )


    cos_pooled_std = np.sqrt(

        0.5
        *
        (
            similarities[
                targets == 1
            ].var()
            +
            similarities[
                targets == 0
            ].var()
        )

        +
        1e-12
    )


    cosine_d_prime = float(
        cos_gap
        /
        cos_pooled_std
    )


    cosine_auc = float(
        roc_auc_score(
            targets,
            similarities
        )
    )


    return {

        "euclidean_auc":
            euclidean_auc,

        "d_pos":
            d_pos,

        "d_neg":
            d_neg,

        "euclidean_gap":
            d_gap,

        "euclidean_d_prime":
            euclidean_d_prime,

        "cosine_auc":
            cosine_auc,

        "cos_pos":
            cos_pos,

        "cos_neg":
            cos_neg,

        "cosine_gap":
            cos_gap,

        "cosine_d_prime":
            cosine_d_prime
    }


# ============================================================
# EVALUATE BOTH MODELS
# ============================================================

euclidean_common_metrics = (
    evaluate_common_pairs(
        euclidean_model,
        common_val_loader
    )
)


cosine_common_metrics = (
    evaluate_common_pairs(
        cosine_model,
        common_val_loader
    )
)


comparison_df = pd.DataFrame(
    [
        {
            "model":
                "Euclidean V3",

            "training_loss":
                "Euclidean Contrastive",

            "margin":
                1.25,

            "val_auc":
                euclidean_common_metrics[
                    "euclidean_auc"
                ],

            "positive_mean":
                euclidean_common_metrics[
                    "d_pos"
                ],

            "negative_mean":
                euclidean_common_metrics[
                    "d_neg"
                ],

            "gap":
                euclidean_common_metrics[
                    "euclidean_gap"
                ],

            "d_prime":
                euclidean_common_metrics[
                    "euclidean_d_prime"
                ],

            "score_type":
                "euclidean distance"
        },

        {
            "model":
                "Cosine V3",

            "training_loss":
                "CosineEmbeddingLoss",

            "margin":
                0.80,

            "val_auc":
                cosine_common_metrics[
                    "cosine_auc"
                ],

            "positive_mean":
                cosine_common_metrics[
                    "cos_pos"
                ],

            "negative_mean":
                cosine_common_metrics[
                    "cos_neg"
                ],

            "gap":
                cosine_common_metrics[
                    "cosine_gap"
                ],

            "d_prime":
                cosine_common_metrics[
                    "cosine_d_prime"
                ],

            "score_type":
                "cosine similarity"
        }
    ]
)


display(
    comparison_df
)


comparison_df.to_csv(

    OUTPUT_DIR
    /
    "euclidean_vs_cosine_common_validation.tsv",

    sep="\t",

    index=False
)

,model,training_loss,margin,val_auc,positive_mean,negative_mean,gap,d_prime,score_type
0,Euclidean V3,Euclidean Contrastive,1.25,0.679463,0.556928,0.690271,0.133342,0.663891,euclidean distance
1,Cosine V3,CosineEmbeddingLoss,0.80,0.662154,0.914038,0.868510,0.045529,0.548053,cosine similarity


In [13]:
# ============================================================
# CELL 13 — SINGLE SAMPLE DATASET
# ============================================================

class SingleSampleDataset(Dataset):

    def __init__(
        self,
        metadata,
        fcgr_memmap,
        id_to_row
    ):

        self.metadata = (
            metadata[
                [
                    "id",
                    "class_id",
                    "disease_clean"
                ]
            ]
            .copy()
            .reset_index(drop=True)
        )

        self.fcgr_memmap = fcgr_memmap


        self.rows = (
            self.metadata["id"]
            .map(id_to_row)
            .to_numpy(dtype=np.int64)
        )


        if np.any(pd.isna(self.rows)):

            raise ValueError(
                "Alcuni sample non hanno una FCGR associata."
            )


    def __len__(self):

        return len(
            self.metadata
        )


    def __getitem__(
        self,
        index
    ):

        row = int(
            self.rows[index]
        )

        x = np.array(
            self.fcgr_memmap[row],
            dtype=np.float32,
            copy=True
        )

        return {

            "x":
                torch.from_numpy(
                    x
                ).unsqueeze(0),

            "id":
                self.metadata.iloc[index]["id"],

            "class_id":
                int(
                    self.metadata.iloc[index]["class_id"]
                ),

            "disease_clean":
                self.metadata.iloc[index]["disease_clean"]
        }

In [14]:
# ============================================================
# CELL 14 — SINGLE SAMPLE DATALOADERS
# ============================================================

train_single_dataset = SingleSampleDataset(

    metadata=
        train_metadata,

    fcgr_memmap=
        fcgr_memmap,

    id_to_row=
        id_to_fcgr_row
)


val_single_dataset = SingleSampleDataset(

    metadata=
        val_pool,

    fcgr_memmap=
        fcgr_memmap,

    id_to_row=
        id_to_fcgr_row
)


train_single_loader = DataLoader(

    train_single_dataset,

    batch_size=
        BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=
        torch.cuda.is_available()
)


val_single_loader = DataLoader(

    val_single_dataset,

    batch_size=
        BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=
        torch.cuda.is_available()
)


print(
    "Train samples:",
    len(train_single_dataset)
)

print(
    "Validation samples:",
    len(val_single_dataset)
)

print(
    "Train batches:",
    len(train_single_loader)
)

print(
    "Validation batches:",
    len(val_single_loader)
)

Train samples: 126265
Validation samples: 12937
Train batches: 987
Validation batches: 102


In [15]:
# ============================================================
# CELL 15 — ESTRAZIONE EMBEDDING
# ============================================================

def extract_embeddings(
    model,
    loader
):

    model.eval()


    embeddings_list = []

    class_ids_list = []

    sample_ids = []

    disease_names = []


    with torch.no_grad():

        for batch in loader:

            x = (
                batch["x"]
                .to(
                    DEVICE,
                    non_blocking=True
                )
            )


            with torch.autocast(

                device_type=
                    DEVICE.type,

                dtype=
                    torch.float16
                    if DEVICE.type == "cuda"
                    else torch.bfloat16,

                enabled=
                    DEVICE.type == "cuda"

            ):

                z = (
                    model.encoder(
                        x
                    )
                )


            embeddings_list.append(
                z
                .float()
                .cpu()
                .numpy()
            )


            class_ids_list.append(
                batch[
                    "class_id"
                ]
                .cpu()
                .numpy()
            )


            sample_ids.extend(
                batch["id"]
            )


            disease_names.extend(
                batch[
                    "disease_clean"
                ]
            )


    embeddings = np.concatenate(
        embeddings_list,
        axis=0
    )


    class_ids = np.concatenate(
        class_ids_list,
        axis=0
    ).astype(
        np.int64
    )


    return {

        "embeddings":
            embeddings,

        "class_ids":
            class_ids,

        "sample_ids":
            np.asarray(
                sample_ids
            ),

        "disease_names":
            np.asarray(
                disease_names
            )
    }

In [16]:
# ============================================================
# CELL 16 — EMBEDDING EXTRACTION
# ============================================================

print(
    "Estrazione Euclidean train..."
)

euclidean_train_embeddings = (
    extract_embeddings(
        euclidean_model,
        train_single_loader
    )
)


print(
    "Estrazione Euclidean validation..."
)

euclidean_val_embeddings = (
    extract_embeddings(
        euclidean_model,
        val_single_loader
    )
)


print(
    "Estrazione Cosine train..."
)

cosine_train_embeddings = (
    extract_embeddings(
        cosine_model,
        train_single_loader
    )
)


print(
    "Estrazione Cosine validation..."
)

cosine_val_embeddings = (
    extract_embeddings(
        cosine_model,
        val_single_loader
    )
)


print()
print("=" * 70)
print("EMBEDDING ESTRATTI")
print("=" * 70)

print(
    "Euclidean train:",
    euclidean_train_embeddings[
        "embeddings"
    ].shape
)

print(
    "Euclidean val:",
    euclidean_val_embeddings[
        "embeddings"
    ].shape
)

print(
    "Cosine train:",
    cosine_train_embeddings[
        "embeddings"
    ].shape
)

print(
    "Cosine val:",
    cosine_val_embeddings[
        "embeddings"
    ].shape
)

Estrazione Euclidean train...
Estrazione Euclidean validation...
Estrazione Cosine train...
Estrazione Cosine validation...

EMBEDDING ESTRATTI
Euclidean train: (126265, 128)
Euclidean val: (12937, 128)
Cosine train: (126265, 128)
Cosine val: (12937, 128)


In [17]:
# ============================================================
# CELL 17 — CLASS PROTOTYPES
# ============================================================

def build_class_prototypes(
    embedding_data
):

    embeddings = (
        embedding_data[
            "embeddings"
        ]
    )

    labels = (
        embedding_data[
            "class_ids"
        ]
    )


    classes = np.array(
        sorted(
            np.unique(
                labels
            )
        ),
        dtype=np.int64
    )


    prototypes = []


    for class_id in classes:

        class_embeddings = (
            embeddings[
                labels == class_id
            ]
        )


        prototype = (
            class_embeddings
            .mean(
                axis=0
            )
        )


        # L2 normalization del prototype
        prototype = (
            prototype
            /
            (
                np.linalg.norm(
                    prototype
                )
                +
                1e-12
            )
        )


        prototypes.append(
            prototype
        )


    prototypes = np.stack(
        prototypes,
        axis=0
    ).astype(
        np.float32
    )


    return (
        classes,
        prototypes
    )


(
    euclidean_classes,
    euclidean_prototypes
) = build_class_prototypes(
    euclidean_train_embeddings
)


(
    cosine_classes,
    cosine_prototypes
) = build_class_prototypes(
    cosine_train_embeddings
)


assert np.array_equal(
    euclidean_classes,
    cosine_classes
)


print(
    "Numero prototipi:",
    len(
        euclidean_classes
    )
)

print(
    "Shape prototipi:",
    euclidean_prototypes.shape
)


prototype_table = pd.DataFrame(
    {
        "class_id":
            euclidean_classes,

        "disease":
            [
                class_id_to_name[
                    int(class_id)
                ]

                for class_id
                in euclidean_classes
            ]
    }
)


display(
    prototype_table
)

Numero prototipi: 18
Shape prototipi: (18, 128)


,class_id,disease
0,0,gastric cancer
1,1,healthy
2,2,ovarian cancer
3,3,prostate cancer
4,4,colorectal cancer
5,5,lymphoma
6,6,hiv infectious disease
7,7,cervical adenocarcinoma
8,8,leukemia
9,9,primary pulmonary hypertension


In [18]:
# ============================================================
# CELL 18 — PROTOTYPE CLASSIFICATION
# ============================================================

def classify_with_prototypes(
    embedding_data,
    classes,
    prototypes,
    metric
):

    embeddings = (
        embedding_data[
            "embeddings"
        ].astype(
            np.float32
        )
    )


    y_true = (
        embedding_data[
            "class_ids"
        ]
    )


    if metric == "euclidean":

        # shape:
        # [samples, classes]

        distances = np.sqrt(
            (
                (
                    embeddings[:, None, :]
                    -
                    prototypes[None, :, :]
                )
                ** 2
            )
            .sum(axis=2)
        )


        prediction_index = (
            distances.argmin(
                axis=1
            )
        )


        y_pred = (
            classes[
                prediction_index
            ]
        )


        scores = (
            -distances
        )


    elif metric == "cosine":

        # Embedding e prototype sono già
        # L2-normalizzati.

        similarities = (
            embeddings
            @
            prototypes.T
        )


        prediction_index = (
            similarities.argmax(
                axis=1
            )
        )


        y_pred = (
            classes[
                prediction_index
            ]
        )


        scores = (
            similarities
        )


    else:

        raise ValueError(
            "metric deve essere "
            "'euclidean' oppure 'cosine'."
        )


    return {

        "y_true":
            y_true,

        "y_pred":
            y_pred,

        "scores":
            scores,

        "prediction_index":
            prediction_index
    }


euclidean_val_predictions = (
    classify_with_prototypes(

        euclidean_val_embeddings,

        euclidean_classes,

        euclidean_prototypes,

        metric="euclidean"
    )
)


cosine_val_predictions = (
    classify_with_prototypes(

        cosine_val_embeddings,

        cosine_classes,

        cosine_prototypes,

        metric="cosine"
    )
)

In [19]:
# ============================================================
# CELL 19 — MULTICLASS VALIDATION METRICS
# ============================================================

def compute_multiclass_metrics(
    predictions
):

    y_true = (
        predictions[
            "y_true"
        ]
    )

    y_pred = (
        predictions[
            "y_pred"
        ]
    )


    return {

        "accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "macro_f1":
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                y_pred
            )
    }


euclidean_multiclass_metrics = (
    compute_multiclass_metrics(
        euclidean_val_predictions
    )
)


cosine_multiclass_metrics = (
    compute_multiclass_metrics(
        cosine_val_predictions
    )
)


multiclass_comparison_df = pd.DataFrame(
    [
        {
            "model":
                "Euclidean V3",

            **euclidean_multiclass_metrics
        },

        {
            "model":
                "Cosine V3",

            **cosine_multiclass_metrics
        }
    ]
)


display(
    multiclass_comparison_df
)


multiclass_comparison_df.to_csv(

    OUTPUT_DIR
    /
    "prototype_validation_comparison.tsv",

    sep="\t",

    index=False
)

,model,accuracy,macro_f1,balanced_accuracy
0,Euclidean V3,0.236995,0.187323,0.243257
1,Cosine V3,0.171601,0.098044,0.184421


In [20]:
# ============================================================
# CELL 20 — PERFORMANCE PER CLASSE
# ============================================================

def per_class_metrics(
    predictions,
    classes,
    class_id_to_name
):

    y_true = predictions[
        "y_true"
    ]

    y_pred = predictions[
        "y_pred"
    ]


    rows = []


    for class_id in classes:

        mask = (
            y_true
            ==
            class_id
        )


        n_samples = int(
            mask.sum()
        )


        correct = int(
            (
                y_pred[mask]
                ==
                class_id
            ).sum()
        )


        recall = (
            correct
            /
            n_samples
            if n_samples > 0
            else np.nan
        )


        rows.append(
            {
                "class_id":
                    int(class_id),

                "disease":
                    class_id_to_name[
                        int(class_id)
                    ],

                "n_val":
                    n_samples,

                "correct":
                    correct,

                "accuracy_per_class":
                    recall
            }
        )


    return (
        pd.DataFrame(
            rows
        )
        .sort_values(
            "accuracy_per_class",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )


euclidean_per_class_df = (
    per_class_metrics(
        euclidean_val_predictions,
        euclidean_classes,
        class_id_to_name
    )
)


cosine_per_class_df = (
    per_class_metrics(
        cosine_val_predictions,
        cosine_classes,
        class_id_to_name
    )
)


print("EUCLIDEAN")
display(
    euclidean_per_class_df
)


print()
print("COSINE")
display(
    cosine_per_class_df
)

EUCLIDEAN


,class_id,disease,n_val,correct,accuracy_per_class
0,9,primary pulmonary hypertension,748,564,0.754011
1,13,melanoma,438,297,0.678082
2,5,lymphoma,1000,442,0.442000
3,0,gastric cancer,1000,427,0.427000
4,17,branchio-oculo-facial syndrome (bofs),205,81,0.395122
5,2,ovarian cancer,1000,390,0.390000
6,7,cervical adenocarcinoma,1000,289,0.289000
7,14,dilated cardiomyopathy,287,60,0.209059
8,15,hypopharynx cancer,225,47,0.208889
9,4,colorectal cancer,1000,169,0.169000



COSINE


,class_id,disease,n_val,correct,accuracy_per_class
0,9,primary pulmonary hypertension,748,677,0.905080
1,13,melanoma,438,323,0.737443
2,5,lymphoma,1000,543,0.543000
3,17,branchio-oculo-facial syndrome (bofs),205,107,0.521951
4,7,cervical adenocarcinoma,1000,268,0.268000
5,0,gastric cancer,1000,193,0.193000
6,16,chronic kidney disease,192,6,0.031250
7,2,ovarian cancer,1000,26,0.026000
8,3,prostate cancer,1000,20,0.020000
9,8,leukemia,1000,19,0.019000


In [21]:
# ============================================================
# CELL 21 — CLASS DISTRIBUTION
# ============================================================

train_counts = (
    train_metadata[
        "class_id"
    ]
    .value_counts()
    .sort_index()
)


val_counts = (
    val_pool[
        "class_id"
    ]
    .value_counts()
    .sort_index()
)


class_distribution_df = pd.DataFrame(
    {
        "class_id":
            sorted(
                class_id_to_name.keys()
            )
    }
)


class_distribution_df[
    "disease"
] = (

    class_distribution_df[
        "class_id"
    ]
    .map(
        class_id_to_name
    )
)


class_distribution_df[
    "n_train"
] = (

    class_distribution_df[
        "class_id"
    ]
    .map(
        train_counts
    )
    .fillna(0)
    .astype(int)
)


class_distribution_df[
    "n_val"
] = (

    class_distribution_df[
        "class_id"
    ]
    .map(
        val_counts
    )
    .fillna(0)
    .astype(int)
)


display(
    class_distribution_df
)

,class_id,disease,n_train,n_val
0,0,gastric cancer,10000,1000
1,1,healthy,10000,1000
2,2,ovarian cancer,10000,1000
3,3,prostate cancer,10000,1000
4,4,colorectal cancer,10000,1000
5,5,lymphoma,10000,1000
6,6,hiv infectious disease,10000,1000
7,7,cervical adenocarcinoma,10000,1000
8,8,leukemia,10000,1000
9,9,primary pulmonary hypertension,6987,748


In [22]:
# ============================================================
# CELL 22 — CONFUSION MATRIX
# ============================================================

def confusion_dataframe(
    predictions,
    classes,
    class_id_to_name
):

    cm = confusion_matrix(
        predictions[
            "y_true"
        ],
        predictions[
            "y_pred"
        ],
        labels=classes
    )


    names = [
        class_id_to_name[
            int(c)
        ]
        for c in classes
    ]


    return pd.DataFrame(
        cm,
        index=names,
        columns=names
    )


euclidean_cm_df = (
    confusion_dataframe(
        euclidean_val_predictions,
        euclidean_classes,
        class_id_to_name
    )
)


display(
    euclidean_cm_df
)

,gastric cancer,healthy,ovarian cancer,prostate cancer,colorectal cancer,lymphoma,hiv infectious disease,cervical adenocarcinoma,leukemia,primary pulmonary hypertension,cataract,hypopharyngeal squamous cell carcinoma,glioblastoma cancer,melanoma,dilated cardiomyopathy,hypopharynx cancer,chronic kidney disease,branchio-oculo-facial syndrome (bofs)
gastric cancer,427,10,1,16,48,45,29,125,29,0,1,10,18,69,13,55,63,41
healthy,64,13,57,21,18,39,13,42,17,191,62,13,7,245,48,43,23,84
ovarian cancer,7,1,390,35,27,202,18,39,0,154,16,11,10,4,3,38,24,21
prostate cancer,22,4,221,45,44,195,22,77,6,150,35,11,12,27,11,52,33,33
colorectal cancer,116,4,17,38,169,199,15,239,14,0,3,7,16,37,6,59,42,19
lymphoma,58,1,81,31,50,442,25,133,5,27,3,6,1,2,2,17,79,37
hiv infectious disease,72,8,53,14,15,84,60,53,36,21,18,15,21,20,32,39,79,360
cervical adenocarcinoma,172,2,41,17,88,249,22,289,15,9,0,0,3,5,4,9,47,28
leukemia,97,7,27,16,21,62,42,52,42,19,18,9,15,211,38,19,53,252
primary pulmonary hypertension,0,0,52,8,0,7,4,1,0,564,36,0,2,6,20,20,6,22


In [23]:
# ============================================================
# CELL 23 — BALANCED REFERENCE BANK
# ============================================================

N_REFERENCES_PER_CLASS = 1500

REFERENCE_SEED = (
    RANDOM_STATE
    +
    70_000
)


def build_balanced_reference_bank(
    embedding_data,
    classes,
    n_per_class=1500,
    seed=42
):

    rng = np.random.default_rng(
        seed
    )


    embeddings = (
        embedding_data[
            "embeddings"
        ]
        .astype(np.float32)
    )


    labels = (
        embedding_data[
            "class_ids"
        ]
        .astype(np.int64)
    )


    reference_embeddings = []

    reference_labels = []

    class_slices = {}


    start = 0


    for class_id in classes:

        class_indices = np.where(
            labels == class_id
        )[0]


        if (
            len(class_indices)
            <
            n_per_class
        ):

            raise ValueError(
                f"Classe {class_id}: "
                f"solo {len(class_indices)} sample, "
                f"ma ne sono richiesti {n_per_class}."
            )


        selected_indices = rng.choice(
            class_indices,
            size=n_per_class,
            replace=False
        )


        class_embeddings = (
            embeddings[
                selected_indices
            ]
        )


        # Normalizzazione di sicurezza
        class_embeddings = (
            class_embeddings
            /
            (
                np.linalg.norm(
                    class_embeddings,
                    axis=1,
                    keepdims=True
                )
                +
                1e-12
            )
        )


        reference_embeddings.append(
            class_embeddings
        )


        reference_labels.append(
            np.full(
                n_per_class,
                class_id,
                dtype=np.int64
            )
        )


        end = (
            start
            +
            n_per_class
        )


        class_slices[
            int(class_id)
        ] = (
            start,
            end
        )


        start = end


    reference_embeddings = np.concatenate(
        reference_embeddings,
        axis=0
    )


    reference_labels = np.concatenate(
        reference_labels,
        axis=0
    )


    return {
        "embeddings":
            reference_embeddings,

        "labels":
            reference_labels,

        "class_slices":
            class_slices
    }


euclidean_reference_bank = (
    build_balanced_reference_bank(

        embedding_data=
            euclidean_train_embeddings,

        classes=
            euclidean_classes,

        n_per_class=
            N_REFERENCES_PER_CLASS,

        seed=
            REFERENCE_SEED
    )
)


print(
    "Reference per classe:",
    N_REFERENCES_PER_CLASS
)

print(
    "Numero classi:",
    len(euclidean_classes)
)

print(
    "Reference totali:",
    len(
        euclidean_reference_bank[
            "embeddings"
        ]
    )
)

print(
    "Shape:",
    euclidean_reference_bank[
        "embeddings"
    ].shape
)

Reference per classe: 1500
Numero classi: 18
Reference totali: 27000
Shape: (27000, 128)


In [24]:
# ============================================================
# CELL 24 — REFERENCE BANK CLASSIFICATION
# ============================================================

K_VALUES = [
    1,
    3,
    5,
    10,
    20,
    50
]


def classify_reference_bank(
    query_embedding_data,
    reference_bank,
    classes,
    k_values,
    batch_size=256
):

    query_embeddings = (
        query_embedding_data[
            "embeddings"
        ]
        .astype(np.float32)
    )


    # Normalizzazione di sicurezza
    query_embeddings = (
        query_embeddings
        /
        (
            np.linalg.norm(
                query_embeddings,
                axis=1,
                keepdims=True
            )
            +
            1e-12
        )
    )


    y_true = (
        query_embedding_data[
            "class_ids"
        ]
        .astype(np.int64)
    )


    reference_embeddings = (
        reference_bank[
            "embeddings"
        ]
        .astype(np.float32)
    )


    class_slices = (
        reference_bank[
            "class_slices"
        ]
    )


    reference_tensor = (
        torch.from_numpy(
            reference_embeddings
        )
        .to(
            DEVICE
        )
    )


    max_k = max(
        k_values
    )


    predictions_by_k = {
        k: []
        for k in k_values
    }


    scores_by_k = {
        k: []
        for k in k_values
    }


    for start_query in range(
        0,
        len(query_embeddings),
        batch_size
    ):

        end_query = min(
            start_query
            +
            batch_size,

            len(query_embeddings)
        )


        query_batch = (
            torch.from_numpy(
                query_embeddings[
                    start_query:
                    end_query
                ]
            )
            .to(
                DEVICE
            )
        )


        # ----------------------------------------------------
        # Similarità con TUTTA la reference bank
        #
        # embeddings L2 normalized:
        # dot product = cosine similarity
        # ----------------------------------------------------

        similarity_matrix = (
            query_batch
            @
            reference_tensor.T
        )


        # score shape:
        # [batch, n_classes]
        class_scores_by_k = {

            k:
                torch.empty(
                    (
                        len(query_batch),
                        len(classes)
                    ),
                    device=DEVICE,
                    dtype=torch.float32
                )

            for k in k_values
        }


        # ----------------------------------------------------
        # Per ogni classe:
        # top-k reference più simili
        # ----------------------------------------------------

        for class_position, class_id in enumerate(
            classes
        ):

            (
                ref_start,
                ref_end
            ) = class_slices[
                int(class_id)
            ]


            class_similarities = (
                similarity_matrix[
                    :,
                    ref_start:
                    ref_end
                ]
            )


            top_values = torch.topk(
                class_similarities,
                k=max_k,
                dim=1,
                largest=True,
                sorted=True
            ).values


            cumulative = torch.cumsum(
                top_values,
                dim=1
            )


            for k in k_values:

                mean_top_k = (
                    cumulative[
                        :,
                        k - 1
                    ]
                    /
                    float(k)
                )


                class_scores_by_k[
                    k
                ][
                    :,
                    class_position
                ] = mean_top_k


        # ----------------------------------------------------
        # Prediction
        # ----------------------------------------------------

        for k in k_values:

            scores = (
                class_scores_by_k[
                    k
                ]
            )


            predicted_position = (
                scores.argmax(
                    dim=1
                )
            )


            predictions = (
                torch.as_tensor(
                    classes,
                    device=DEVICE
                )[
                    predicted_position
                ]
            )


            predictions_by_k[
                k
            ].append(
                predictions
                .cpu()
                .numpy()
            )


            scores_by_k[
                k
            ].append(
                scores
                .cpu()
                .numpy()
            )


        del similarity_matrix


    # ========================================================
    # CONCAT
    # ========================================================

    results = {}


    for k in k_values:

        results[k] = {

            "y_true":
                y_true,

            "y_pred":
                np.concatenate(
                    predictions_by_k[
                        k
                    ]
                ),

            "scores":
                np.concatenate(
                    scores_by_k[
                        k
                    ],
                    axis=0
                )
        }


    del reference_tensor

    torch.cuda.empty_cache()


    return results

In [25]:
# ============================================================
# CELL 25 — KNN VALIDATION
# ============================================================

euclidean_knn_results = (
    classify_reference_bank(

        query_embedding_data=
            euclidean_val_embeddings,

        reference_bank=
            euclidean_reference_bank,

        classes=
            euclidean_classes,

        k_values=
            K_VALUES,

        batch_size=
            256
    )
)


knn_rows = []


for k in K_VALUES:

    predictions = (
        euclidean_knn_results[
            k
        ]
    )


    y_true = (
        predictions[
            "y_true"
        ]
    )

    y_pred = (
        predictions[
            "y_pred"
        ]
    )


    knn_rows.append(
        {
            "k":
                k,

            "accuracy":
                accuracy_score(
                    y_true,
                    y_pred
                ),

            "macro_f1":
                f1_score(
                    y_true,
                    y_pred,
                    average="macro",
                    zero_division=0
                ),

            "balanced_accuracy":
                balanced_accuracy_score(
                    y_true,
                    y_pred
                )
        }
    )


euclidean_knn_validation_df = (
    pd.DataFrame(
        knn_rows
    )
    .sort_values(
        "macro_f1",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


display(
    euclidean_knn_validation_df
)

,k,accuracy,macro_f1,balanced_accuracy
0,10,0.253691,0.223264,0.247710
1,20,0.257710,0.222750,0.251638
2,50,0.263585,0.222650,0.257669
3,5,0.242792,0.219737,0.239958
4,3,0.224163,0.206176,0.222926
5,1,0.169050,0.159526,0.169765


In [26]:
# ============================================================
# CELL 26 — PROTOTYPE VS REFERENCE BANK
# ============================================================

best_knn_row = (
    euclidean_knn_validation_df
    .iloc[0]
)


BEST_K = int(
    best_knn_row[
        "k"
    ]
)


prototype_vs_knn_df = pd.DataFrame(
    [
        {
            "method":
                "Single prototype",

            "k":
                np.nan,

            "accuracy":
                euclidean_multiclass_metrics[
                    "accuracy"
                ],

            "macro_f1":
                euclidean_multiclass_metrics[
                    "macro_f1"
                ],

            "balanced_accuracy":
                euclidean_multiclass_metrics[
                    "balanced_accuracy"
                ]
        },

        {
            "method":
                "Reference bank",

            "k":
                BEST_K,

            "accuracy":
                best_knn_row[
                    "accuracy"
                ],

            "macro_f1":
                best_knn_row[
                    "macro_f1"
                ],

            "balanced_accuracy":
                best_knn_row[
                    "balanced_accuracy"
                ]
        }
    ]
)


display(
    prototype_vs_knn_df
)


print()
print(
    "Best k:",
    BEST_K
)

,method,k,accuracy,macro_f1,balanced_accuracy
0,Single prototype,NaN,0.236995,0.187323,0.243257
1,Reference bank,10.0,0.253691,0.223264,0.247710



Best k: 10
